# 05. Modelado con XGBoost

Entrena clasificadores para horizontes de 7, 5 y 3 días, evalúa distintos thresholds, guarda métricas y persiste un modelo por horizonte.


In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

DATA_RAW_DIR = PROJECT_ROOT / "data" / "raw"
DATA_PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
MODELS_DIR = PROJECT_ROOT / "models"
PLOTS_DIR = PROJECT_ROOT / "outputs" / "plots"
METRICS_DIR = PROJECT_ROOT / "outputs" / "metrics"

for directory in [DATA_PROCESSED_DIR, MODELS_DIR, PLOTS_DIR, METRICS_DIR]:
    directory.mkdir(parents=True, exist_ok=True)


In [2]:
import pickle

import pandas as pd
from IPython.display import display

from src.evaluation import evaluate_model, plot_precision_recall, save_metrics
from src.feature_engineering import DEFAULT_FEATURE_COLUMNS
from src.modeling import train_xgboost_model


In [3]:
features_df = pd.read_parquet(DATA_PROCESSED_DIR / "features.parquet")
X = features_df[DEFAULT_FEATURE_COLUMNS].fillna(0)
PERSISTED_MODEL_USE_SMOTE = True
summary_rows = []

for window in [7, 5, 3]:
    target_column = f"correctivo_prox_{window}d"
    y = features_df[target_column].astype(int)

    for use_smote in [False, True]:
        run_name = f"xgb_{window}d_{'con_smote' if use_smote else 'sin_smote'}"
        training_artifacts = train_xgboost_model(X, y, use_smote=use_smote)
        metrics = evaluate_model(
            training_artifacts["y_test"],
            training_artifacts["y_score"],
            thresholds=(0.3, 0.4, 0.5, 0.6),
        )
        metrics["window_days"] = window
        metrics["use_smote"] = use_smote
        metrics["feature_columns"] = DEFAULT_FEATURE_COLUMNS
        metrics["scale_pos_weight"] = training_artifacts["scale_pos_weight"]

        save_metrics(metrics, METRICS_DIR / f"{run_name}.json")
        plot_precision_recall(
            training_artifacts["y_test"],
            training_artifacts["y_score"],
            PLOTS_DIR / f"{run_name}_pr_curve.png",
            title=f"PR Curve - {run_name}",
        )

        for threshold, threshold_metrics in metrics["threshold_metrics"].items():
            positive_metrics = threshold_metrics["classification_report"].get("1", {})
            summary_rows.append(
                {
                    "run_name": run_name,
                    "window_days": window,
                    "use_smote": use_smote,
                    "threshold": float(threshold),
                    "precision_class_1": positive_metrics.get("precision"),
                    "recall_class_1": positive_metrics.get("recall"),
                    "f1_class_1": positive_metrics.get("f1-score"),
                    "support_class_1": positive_metrics.get("support"),
                }
            )

        if use_smote == PERSISTED_MODEL_USE_SMOTE:
            with (MODELS_DIR / f"xgb_{window}d.pkl").open("wb") as file:
                pickle.dump(training_artifacts["model"], file)

summary_df = pd.DataFrame(summary_rows).sort_values(
    ["window_days", "use_smote", "threshold"],
    ascending=[True, True, True],
)
summary_df.to_csv(METRICS_DIR / "model_summary.csv", index=False)

display(summary_df[summary_df["threshold"] == 0.5])
print("Modelos guardados en:", MODELS_DIR)
print("Métricas guardadas en:", METRICS_DIR)


,run_name,window_days,use_smote,threshold,precision_class_1,recall_class_1,f1_class_1,support_class_1
18,xgb_3d_sin_smote,3,False,0.5,0.621128,0.645747,0.633198,1211.0
22,xgb_3d_con_smote,3,True,0.5,0.625990,0.652353,0.638900,1211.0
10,xgb_5d_sin_smote,5,False,0.5,0.739432,0.698756,0.718519,1527.0
14,xgb_5d_con_smote,5,True,0.5,0.716686,0.810085,0.760529,1527.0
2,xgb_7d_sin_smote,7,False,0.5,0.836852,0.753022,0.792727,1737.0
6,xgb_7d_con_smote,7,True,0.5,0.792261,0.895797,0.840854,1737.0


Modelos guardados en: C:\Users\josep\Desktop\repo_mantenimiento_predictivo\models
Métricas guardadas en: C:\Users\josep\Desktop\repo_mantenimiento_predictivo\outputs\metrics
